# Extracció dades amb Claude
extracció de dades json del excel amb claude

!pip install anthropic

In [3]:
import pandas as pd
import time
import os
import json
import re
from anthropic import Anthropic
from getpass import getpass



- Solicitar la clave de Anthropic de forma segura

In [ ]:
password = getpass("Introduce tu API KEY de Anthropic (Claude): ")

1. CONFIGURACIÓN Y LIMPIEZA DE CREDENCIALES
- Inicializamos el cliente oficial de Anthropic
- Columnas de origen y destino
- Asegurar que todas las columnas existen en excel
- Patrones locales para la estandarizacion posterior de materiales

2. Bucle completo para todo el documento
- MODIFICACIÓN: Saltamos de la fila 1 a la 269 (índices 0 a 268)
- Solo imprimimos las primeras y las últimas que se saltan para no saturar la pantalla
- Mapeo seguro al DataFrame
- LIMPIEZA ADICIONAL DEL TACO
- Clasificación de materiales local

3. GUARDADO SEGURO FINAL




In [ ]:

API_KEY = password.strip()  
archivo_unico = "zapatillas_trail_REVISADO_CORREGIDO.xlsx"

if not os.path.exists(archivo_unico):
    print(f"❌ No se encontró el archivo {archivo_unico}")
    exit()

client = Anthropic(api_key=API_KEY)
df = pd.read_excel(archivo_unico)

columnas_origen = ['Pros_Contras', 'Conclusiones', 'Tipo_Corredor', 'Distancia', 'Tipo_Terreno', 'Durabilidad']
columnas_excel_destino = [
    'Upper', 'upper_categoria', 
    'cordones', 
    'mediasuela', 'mediasuela_categoria',
    'suela', 'suela_categoria',
    'taco', 'nueva_terreno', 'terreno_categoria',
    'nueva_distancia', 'distancia_categoria',
    'nueva_tipo de corredor', 'corredor_categoria',
    'nueva_durabilidad', 'durabilidad_categoria'
]

for col in columnas_excel_destino:
    if col not in df.columns:
        df[col] = float('nan')

diccionario_mapeo = {
    'upper': 'Upper',
    'cordones': 'cordones',
    'mediasuela': 'mediasuela',
    'suela': 'suela',
    'taco': 'taco',
    'nueva_terreno': 'nueva_terreno',
    'terreno_categoria': 'terreno_categoria',
    'nueva_distancia': 'nueva_distancia',
    'distancia_categoria': 'distancia_categoria',
    'nueva_tipo de corredor': 'nueva_tipo de corredor',
    'corredor_categoria': 'corredor_categoria',
    'nueva_durabilidad': 'nueva_durabilidad',
    'durabilidad_categoria': 'durabilidad_categoria'
}

patrones_suela = ['vibram', 'frixion', 'asicsgrip', 'continental', 'contagrip', 'pomoca', 'michelin', 'pwrtrac', 'surface ctrl', 'optivibe']
patrones_mediasuela = ['eva', 'peba', 'tpu', 'nitrogeno', 'infinitoo', 'superfoam', 'egomax', 'floatpro', 'boost', 'flytefoam', 'energy surge']
patrones_upper = ['matryx', 'mesh', 'jacquard', 'knit', 'kevlar', 'gore-tex', 'cordura', 'tejido']

def normalizar_material(texto_ia, patrones_lista):
    if not texto_ia or pd.isna(texto_ia):
        return 'Otros / No especificado'
    texto_min = str(texto_ia).lower()
    for patron in patrones_lista:
        if patron in texto_min:
            return patron.capitalize()
    palabras = [p.strip() for p in texto_min.replace(',', ' ').split() if len(p.strip()) > 3]
    if palabras:
        return palabras[0].capitalize() 
    return 'Otros / No especificado'

def limpiar_taco(valor):
    """Extrae solo el número del taco o devuelve NaN si no hay datos numéricos válidos."""
    if not valor or pd.isna(valor):
        return float('nan')
    
    texto = str(valor).strip().replace(',', '.')
    match = re.search(r'\d+\.\d+|\d+', texto)
    if match:
        num_str = match.group()
        try:
            return float(num_str) if '.' in num_str else int(num_str)
        except ValueError:
            return float('nan')
    return float('nan')

total_filas = len(df)
print(f"📖 Archivo cargado correctamente ({total_filas} filas). Iniciando el procesamiento desde la fila 270 hasta el final...")

for idx, fila in df.iterrows():
    if idx < 269:
        if idx == 0 or idx == 268:
            print(f"⏭️ Saltando fila [{idx + 1}]: {fila['Modelo']} (Ya procesada)")
        continue
        
    modelo = fila['Modelo']
    print(f"\n==================================================")
    print(f"🧠 PROCESANDO FILA [{idx + 1} / {total_filas}]: {modelo}")
    print(f"==================================================")
    
    fragmentos_texto = []
    for col in columnas_origen:
        if col in df.columns:
            valor_celda = str(fila[col]).strip()
            if valor_celda and valor_celda.lower() != 'nan':
                fragmentos_texto.append(f"[{col.upper()}]: {valor_celda}")

    texto_combinado = "\n".join(fragmentos_texto)
    
    if not texto_combinado.strip():
        print(f"⚠️ Saltando {modelo}: Sin datos de origen en las columnas de texto.")
        continue

    prompt = f"""
    Analiza minuciosamente los textos de esta zapatilla de trail running para extraer especificaciones técnicas.

    <texto_origen>
    {texto_combinado}
    </texto_origen>

    <reglas_categorias>
    - terreno_categoria: Elije una opción exacta de: "Fácil/Rodador", "Polivalente", "Barro/Blando", "Compacto/Roca", "Técnico".
    - distancia_categoria: Elije una opción exacta de: "Corta", "Media", "Maratón", "Larga", "Ultra".
    - corredor_categoria: Elije una opción exacta de: "Ligero", "Medio", "Pesado".
    - durabilidad_categoria: Elije una opción exacta de: "Alta", "Media", "Baja".
    </reglas_categorias>

    Genera tu respuesta final metida estrictamente dentro de etiquetas <json_output></json_output>. 
    El contenido interno debe ser un objeto JSON plano que use exactamente esta estructura:
    {{
        "upper": "Material o tecnología del upper",
        "cordones": "Tipo de cierre",
        "mediasuela": "Nombre de la espuma",
        "suela": "Compuesto de la suela",
        "taco": "Prominencia en mm (ej: 4mm o No especificado)",
        "nueva_terreno": "Descripción cualitativa del terreno óptimo",
        "terreno_categoria": "Categoría fija de terreno",
        "nueva_distancia": "Descripción cualitativa de la distancia ideal",
        "distancia_categoria": "Categoría fija de distancia",
        "nueva_tipo de corredor": "Descripción cualitativa del tipo de corredor",
        "corredor_categoria": "Categoría fija de corredor",
        "nueva_durabilidad": "Descripción cualitativa de la durabilidad",
        "durabilidad_categoria": "Categoría fija de durabilidad"
    }}
    """

    try:
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            temperature=0.0,
            system="Eres un autómata experto en análisis de datos. Tu única función es extraer datos técnicos de calzado y responder usando bloques XML <json_output>.",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        
        respuesta_raw = response.content[0].text.strip()
        
        match = re.search(r'<json_output>(.*?)</json_output>', respuesta_raw, re.DOTALL)
        if match:
            json_puro = match.group(1).strip()
        else:
            json_puro = respuesta_raw
            
        datos_extraidos = json.loads(json_puro)
        
        for llave_json, columna_excel in diccionario_mapeo.items():
            valor_extraido = datos_extraidos.get(llave_json)
            if valor_extraido and str(valor_extraido).lower() not in ['null', 'none', 'nan', '']:
                df.at[idx, columna_excel] = str(valor_extraido).strip()
            else:
                df.at[idx, columna_excel] = float('nan')

        taco_original = df.at[idx, 'taco']
        df.at[idx, 'taco'] = limpiar_taco(taco_original)

        texto_suela = df.at[idx, 'suela']
        cat_suela = normalizar_material(texto_suela, patrones_suela)
        df.at[idx, 'suela_categoria'] = cat_suela
        
        texto_mediasuela = df.at[idx, 'mediasuela']
        cat_mediasuela = normalizar_material(texto_mediasuela, patrones_mediasuela)
        df.at[idx, 'mediasuela_categoria'] = cat_mediasuela
        
        texto_upper = df.at[idx, 'Upper']
        cat_upper = normalizar_material(texto_upper, patrones_upper)
        df.at[idx, 'upper_categoria'] = cat_upper

        print(f"   ✓ Extraído con éxito | Taco limpio: {df.at[idx, 'taco']} | Terreno: {df.at[idx, 'terreno_categoria']}")
        
    except Exception as e:
        print(f"❌ Error crítico en posición {idx + 1} ({modelo}): {e}")
        
    time.sleep(1.2) 

print(f"\n💾 Guardando los nuevos datos procesados a partir de la fila 270 en {archivo_unico}...")
df.to_excel(archivo_unico, index=False)

print("==================================================")
print(f"🎉 ¡PROCESAMIENTO DESDE FILA 270 HASTA EL FINAL COMPLETADO!")
print("==================================================")

📖 Archivo cargado correctamente (935 filas). Iniciando el procesamiento desde la fila 270 hasta el final...
⏭️ Saltando fila [1]: Terrex Agravic Speed Ultra 2 (Ya procesada)
⏭️ Saltando fila [269]: Wave Ibuki 3 (Ya procesada)

🧠 PROCESANDO FILA [270 / 935]: Fuga Pro 3
   ✓ Extraído con éxito | Taco limpio: 4 | Terreno: Polivalente

🧠 PROCESANDO FILA [271 / 935]: Fresh Foam More Trail v1
   ✓ Extraído con éxito | Taco limpio: nan | Terreno: Fácil/Rodador

🧠 PROCESANDO FILA [272 / 935]: Fresh Foam Hierro v5 GTX
   ✓ Extraído con éxito | Taco limpio: nan | Terreno: Polivalente

🧠 PROCESANDO FILA [273 / 935]: S-Lab Cross
   ✓ Extraído con éxito | Taco limpio: 5 | Terreno: Barro/Blando

🧠 PROCESANDO FILA [274 / 935]: Cascadia 15
   ✓ Extraído con éxito | Taco limpio: 4 | Terreno: Polivalente

🧠 PROCESANDO FILA [275 / 935]: Sky Pro
   ✓ Extraído con éxito | Taco limpio: nan | Terreno: Técnico

🧠 PROCESANDO FILA [276 / 935]: MT Cushion
   ✓ Extraído con éxito | Taco limpio: nan | Terreno: Pol